# Section 4: Build and assess a model

This notebook compares Logistic Regression, Random Forest (an ensemble of decision trees), Support Vector Machine, and K-Nearest Neighbors across four expanding chronological validation windows. The selected candidate is refitted on all labelled rows to create `submission.csv`.

The preprocessing recipe below mirrors the decisions in `Preprocessing.ipynb` so this notebook can run in its own kernel. The Decision Tree importance selector follows the final strategy from `03_Feature_Selection.ipynb`; it is fitted within each fold on that fold's training data. Each validation window is later than its fitting data, with a 90-day gap so fitting targets have time to become observable. Rows without dates or inside a gap are excluded from validation, but included when the final model is fitted on all training data.

In [89]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

project_folder = Path.cwd()
if not (project_folder / "data" / "train.csv").is_file():
    project_folder = project_folder.parent
if not (project_folder / "data" / "train.csv").is_file():
    raise FileNotFoundError("Could not find data/train.csv from the notebook's working directory.")

data_folder = project_folder / "data"
train = pd.read_csv(data_folder / "train.csv", index_col="ID")
test = pd.read_csv(data_folder / "test.csv", index_col="ID")
sample_submission = pd.read_csv(data_folder / "sample_submission.csv")

TARGET = "repeat_purchase_90d"
RANDOM_STATE = 42
assert TARGET in train.columns
assert TARGET not in test.columns
assert train.index.is_unique and test.index.is_unique
assert train[TARGET].notna().all()
assert set(train[TARGET].unique()) == {0, 1}
assert set(test.columns) == set(train.drop(columns=TARGET).columns)

X = train.drop(columns=TARGET)
y = train[TARGET].astype(int)
print(f"Training rows: {len(train):,}; test rows: {len(test):,}")
display(y.value_counts(normalize=True).sort_index().rename("target proportion").to_frame())

Training rows: 6,534; test rows: 986


,target proportion
repeat_purchase_90d,
0,0.440771
1,0.559229


## 1. Set up chronological validation

The preliminary split below defines a fitting-period example for the preprocessing schema. Model comparison uses four expanding-window folds: the final four of ten consecutive blocks of distinct purchase dates, with a 90-day gap before each block. All purchases on a date stay together. Rows without dates and rows inside a gap are omitted from validation, but included when the final model is fitted.

In [90]:
purchase_dates = pd.to_datetime(X["purchase_date"], errors="coerce")
sorted_dates = purchase_dates.dropna().sort_values()
split_position = int(len(sorted_dates) * 0.80)
validation_start = sorted_dates.iloc[split_position]
fitting_end = validation_start - pd.Timedelta(days=90)

fitting_rows = purchase_dates < fitting_end
validation_rows = purchase_dates >= validation_start
gap_rows = purchase_dates.notna() & ~fitting_rows & ~validation_rows

X_fit = X.loc[fitting_rows].copy()
X_validation = X.loc[validation_rows].copy()
y_fit = y.loc[fitting_rows]
y_validation = y.loc[validation_rows]

assert not X_fit.empty and not X_validation.empty
assert y_fit.nunique() == 2 and y_validation.nunique() == 2
assert purchase_dates[fitting_rows].max() + pd.Timedelta(days=90) < validation_start

print(f"Validation begins: {validation_start.date()}")
print(f"Fitting rows: {len(X_fit):,}; validation rows: {len(X_validation):,}")
print(f"Rows in the 90-day gap: {gap_rows.sum():,}")
print(f"Rows without a purchase date: {purchase_dates.isna().sum():,}")

Validation begins: 2023-06-17
Fitting rows: 5,024; validation rows: 1,298
Rows in the 90-day gap: 147
Rows without a purchase date: 65


## 2. Prepare features inside each model pipeline

Apply the cleaning and feature decisions from `Preprocessing.ipynb`: replace impossible negative numeric values and infinities with missing values, add missing-history flags and value per detail, clean category labels, and log-transform nonnegative magnitudes. Exclude customer and row identifiers, the anonymized date, and `random_noise` from predictors. Median imputation, scaling, and one-hot category encoding are then learned only from the fitting data.

In [91]:
CATEGORICAL_COLUMNS = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series",
]
NUMERIC_COLUMNS = X.select_dtypes(include="number").columns.tolist()
NUMERIC_COLUMNS = [column for column in NUMERIC_COLUMNS if column not in {"customer_id", "random_noise"}]


def clean_values(data):
    """Replace impossible numeric values with missing values without dropping rows."""
    cleaned = data.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = cleaned[column].replace([np.inf, -np.inf], np.nan)
        cleaned.loc[cleaned[column] < 0, column] = np.nan
    for column in ["prior_mean_purchase_value", "spend_365d"]:
        cleaned.loc[cleaned[column] >= 1_000_000_000, column] = np.nan
    return cleaned


def prepare_features(data):
    """Clean inputs, create simple features, and return model-ready columns."""
    prepared = clean_values(data)
    for column in NUMERIC_COLUMNS:
        prepared[column + "_missing"] = prepared[column].isna().astype(int)

    prepared["first_purchase"] = (prepared["prior_purchase_count"] == 0).astype(int)
    prepared["history_unknown"] = prepared["prior_purchase_count"].isna().astype(int)
    detail_count = prepared["N_ Detalhes"].where(prepared["N_ Detalhes"] > 0)
    prepared["net_value_per_detail"] = prepared["Total Liquido"] / detail_count

    for column in CATEGORICAL_COLUMNS:
        labels = prepared[column].astype("string").str.strip().replace("", pd.NA)
        prepared[column] = labels.to_numpy(dtype=object, na_value=np.nan)

    prepared = prepared.drop(columns=["customer_id", "purchase_date", "random_noise"])
    log_columns = NUMERIC_COLUMNS + ["net_value_per_detail"]
    for column in log_columns:
        prepared[column] = np.log1p(prepared[column])
    return prepared


prepared_fit_example = prepare_features(X_fit)
MODEL_NUMERIC_COLUMNS = prepared_fit_example.select_dtypes(include="number").columns.tolist()
print(f"Numeric features before encoding: {len(MODEL_NUMERIC_COLUMNS)}")
print(f"Categorical features before encoding: {len(CATEGORICAL_COLUMNS)}")

Numeric features before encoding: 59
Categorical features before encoding: 6


In [92]:
def build_preprocessor(scale=True):
    """Create fresh preprocessing steps to be fitted within one training split."""
    numeric_steps = [("fill_missing", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))

    numeric_pipeline = Pipeline(numeric_steps)
    categorical_pipeline = Pipeline([
        ("fill_missing", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    combine_columns = ColumnTransformer([
        ("numeric", numeric_pipeline, MODEL_NUMERIC_COLUMNS),
        ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
    ])
    return Pipeline([
        ("prepare_features", FunctionTransformer(prepare_features)),
        ("combine_columns", combine_columns),
    ])


def build_model_pipeline(classifier, scale=True, feature_selector=None):
    """Combine preprocessing, optional feature selection, and a classifier."""
    steps = [("preprocessing", build_preprocessor(scale=scale))]
    if feature_selector is not None:
        steps.append(("feature_selection", feature_selector))
    steps.append(("classifier", classifier))
    return Pipeline(steps)

## 3. Compare and tune candidate models

The classifier families are Logistic Regression, Random Forest (an ensemble of decision trees), Support Vector Machine (RBF kernel), and K-Nearest Neighbors. The Decision Tree importance selector with a mean threshold is the strategy selected in `03_Feature_Selection.ipynb`; it is refitted inside every fold and applied to every model candidate, except the all-features Logistic Regression baseline. SVM and KNN use scaled numeric features; tree-based estimators do not. ROC AUC is the primary selection metric, using mean class-1 F1 as a tie-breaker; fold variation shows consistency over time.

In [93]:
def build_selected_model_pipeline(classifier, scale=True):
    """Apply the selected tree-importance feature selection before a classifier."""
    return build_model_pipeline(
        classifier,
        scale=scale,
        feature_selector=SelectFromModel(
            DecisionTreeClassifier(random_state=RANDOM_STATE),
            threshold="mean",
        ),
    )


candidate_models = {
    "Logistic Regression": build_model_pipeline(
        LogisticRegression(C=0.1, max_iter=6000, random_state=RANDOM_STATE)
    ),
    "Logistic Regression + tree feature selection": build_selected_model_pipeline(
        LogisticRegression(C=0.1, max_iter=6000, random_state=RANDOM_STATE)
    ),
    "Random Forest + tree feature selection": build_selected_model_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            max_depth=14,
            min_samples_leaf=10,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        scale=False,
    ),
    "SVM (RBF, C=1) + tree feature selection": build_selected_model_pipeline(
        SVC(C=1.0, kernel="rbf", random_state=RANDOM_STATE)
    ),
    "SVM (RBF, C=10) + tree feature selection": build_selected_model_pipeline(
        SVC(C=10.0, kernel="rbf", random_state=RANDOM_STATE)
    ),
    "K-Nearest Neighbors (k=15) + tree feature selection": build_selected_model_pipeline(
        KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1)
    ),
    "K-Nearest Neighbors (k=31) + tree feature selection": build_selected_model_pipeline(
        KNeighborsClassifier(n_neighbors=31, weights="distance", n_jobs=-1)
    ),
}

unique_dates = pd.Index(sorted_dates.unique()).sort_values()
date_blocks = np.array_split(unique_dates.to_numpy(), 10)
validation_blocks = date_blocks[6:]
assert len(validation_blocks) == 4 and all(len(block) > 0 for block in validation_blocks)

def positive_class_metrics(y_true, y_pred):
    """Compute class-1 precision, recall, and F1 directly from TP, FP, and FN."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


validation_results = []
last_fold_models = {}
for fold_number, validation_dates in enumerate(validation_blocks, start=1):
    validation_start = pd.Timestamp(validation_dates[0])
    validation_end = pd.Timestamp(validation_dates[-1])
    fitting_end = validation_start - pd.Timedelta(days=90)
    fitting_rows = purchase_dates < fitting_end
    validation_rows = purchase_dates.between(validation_start, validation_end, inclusive="both")
    X_fold_fit = X.loc[fitting_rows]
    y_fold_fit = y.loc[fitting_rows]
    X_fold_validation = X.loc[validation_rows]
    y_fold_validation = y.loc[validation_rows]

    assert not X_fold_fit.empty and not X_fold_validation.empty
    assert y_fold_fit.nunique() == 2 and y_fold_validation.nunique() == 2
    assert purchase_dates[fitting_rows].max() + pd.Timedelta(days=90) < validation_start

    for model_name, candidate in candidate_models.items():
        fitted_model = clone(candidate).fit(X_fold_fit, y_fold_fit)
        predictions = fitted_model.predict(X_fold_validation)
        scores = (
            fitted_model.predict_proba(X_fold_validation)[:, 1]
            if hasattr(fitted_model, "predict_proba")
            else fitted_model.decision_function(X_fold_validation)
        )
        class_1_metrics = positive_class_metrics(y_fold_validation, predictions)
        validation_results.append({
            "fold": fold_number,
            "model": model_name,
            "validation_start": validation_start.date(),
            "validation_end": validation_end.date(),
            "fitting_rows": len(X_fold_fit),
            "validation_rows": len(X_fold_validation),
            "accuracy": accuracy_score(y_fold_validation, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_fold_validation, predictions),
            "precision": class_1_metrics["precision"],
            "recall": class_1_metrics["recall"],
            "f1": class_1_metrics["f1"],
            "roc_auc": roc_auc_score(y_fold_validation, scores),
        })
        if fold_number == len(validation_blocks):
            last_fold_models[model_name] = fitted_model
            last_validation_rows = validation_rows
            last_validation_start = validation_start
            last_validation_end = validation_end

fold_results = pd.DataFrame(validation_results)
results = (
    fold_results.groupby("model")
    .agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        f1_mean=("f1", "mean"),
    )
    .sort_values(["roc_auc_mean", "f1_mean"], ascending=False, kind="stable")
    .reset_index()
)
display(results.round(3))
print("ROC AUC by chronological validation fold:")
display(
    fold_results.pivot(index="model", columns="fold", values="roc_auc")
    .reindex(results["model"])
    .round(3)
)

feature_selected_results = results[results["model"] != "Logistic Regression"]
assert not feature_selected_results.empty
best_model_name = feature_selected_results.iloc[0]["model"]
best_validation_model = last_fold_models[best_model_name]
X_validation = X.loc[last_validation_rows]
y_validation = y.loc[last_validation_rows]
print(
    "Selected by mean four-fold ROC AUC among feature-selected pipelines "
    f"(F1 tie-breaker): {best_model_name}"
)
print(
    f"Latest validation window: {last_validation_start.date()} to "
    f"{last_validation_end.date()}"
)

,model,roc_auc_mean,roc_auc_std,accuracy_mean,balanced_accuracy_mean,f1_mean
0,Logistic Regression,0.799,0.024,0.724,0.720,0.746
1,Logistic Regression + tree feature selection,0.798,0.023,0.737,0.731,0.762
2,Random Forest + tree feature selection,0.797,0.019,0.716,0.709,0.748
3,"SVM (RBF, C=1) + tree feature selection",0.788,0.031,0.717,0.713,0.744
4,K-Nearest Neighbors (k=31) + tree feature sele...,0.774,0.033,0.707,0.702,0.738
5,K-Nearest Neighbors (k=15) + tree feature sele...,0.764,0.037,0.706,0.701,0.736
6,"SVM (RBF, C=10) + tree feature selection",0.759,0.043,0.701,0.696,0.727


ROC AUC by chronological validation fold:


fold,1,2,3,4
model,,,,
Logistic Regression,0.832,0.773,0.797,0.793
Logistic Regression + tree feature selection,0.824,0.769,0.793,0.804
Random Forest + tree feature selection,0.812,0.770,0.795,0.810
"SVM (RBF, C=1) + tree feature selection",0.811,0.748,0.779,0.815
K-Nearest Neighbors (k=31) + tree feature selection,0.813,0.733,0.767,0.783
K-Nearest Neighbors (k=15) + tree feature selection,0.798,0.713,0.766,0.778
"SVM (RBF, C=10) + tree feature selection",0.779,0.710,0.739,0.809


Selected by mean four-fold ROC AUC among feature-selected pipelines (F1 tie-breaker): Logistic Regression + tree feature selection
Latest validation window: 2024-04-13 to 2025-02-15


In [94]:
feature_selector = best_validation_model.named_steps.get("feature_selection")
if feature_selector is None:
    print(f"{best_model_name} uses all prepared features.")
else:
    feature_names = best_validation_model.named_steps["preprocessing"].named_steps[
        "combine_columns"
    ].get_feature_names_out()
    selected_feature_names = feature_names[feature_selector.get_support()]
    print(f"{best_model_name} selected {len(selected_feature_names)} of {len(feature_names)} encoded features.")
    display(pd.Series(selected_feature_names, name="selected feature").to_frame())

best_predictions = best_validation_model.predict(X_validation)
best_scores = (
    best_validation_model.predict_proba(X_validation)[:, 1]
    if hasattr(best_validation_model, "predict_proba")
    else best_validation_model.decision_function(X_validation)
)
print(f"\nValidation confusion matrix for {best_model_name} (rows=true, columns=predicted):")
display(pd.DataFrame(
    confusion_matrix(y_validation, best_predictions),
    index=["actual 0", "actual 1"],
    columns=["predicted 0", "predicted 1"],
))
class_1_metrics = positive_class_metrics(y_validation, best_predictions)
print("Positive-class metrics calculated from TP, FP, and FN:")
print(
    f"TP={class_1_metrics['tp']}, FP={class_1_metrics['fp']}, "
    f"FN={class_1_metrics['fn']}"
)
print(
    f"Precision={class_1_metrics['precision']:.3f}; "
    f"Recall={class_1_metrics['recall']:.3f}; "
    f"F1=2*(P*R)/(P+R)={class_1_metrics['f1']:.3f}"
)
print(f"Latest-fold ROC AUC: {roc_auc_score(y_validation, best_scores):.3f}")

Logistic Regression + tree feature selection selected 28 of 169 encoded features.


,selected feature
0,numeric__Total a Pagar
1,numeric__Total Bruto
2,numeric__Total Liquido
3,numeric__Total Imp_
4,numeric__Total Desc_ Linha
5,numeric__Custo
6,numeric__N_ Detalhes
7,numeric__days_since_previous_purchase
8,numeric__customer_tenure_days
9,numeric__prior_purchase_count



Validation confusion matrix for Logistic Regression + tree feature selection (rows=true, columns=predicted):


,predicted 0,predicted 1
actual 0,164,72
actual 1,76,286


Positive-class metrics calculated from TP, FP, and FN:
TP=286, FP=72, FN=76
Precision=0.799; Recall=0.790; F1=2*(P*R)/(P+R)=0.794
Latest-fold ROC AUC: 0.804


### Interpreting the comparison

For class 1, precision is P = TP / (TP + FP), recall is R = TP / (TP + FN), and F1 is computed explicitly as 2 * (P * R) / (P + R). If a denominator is zero, the corresponding metric is set to zero. ROC AUC evaluates ranking over thresholds, while balanced accuracy gives equal weight to both classes. Repeated chronological folds show whether a model performs consistently over time, but using these folds to compare candidates can make the winning mean optimistic. Confirm that Kaggle uses the same positive class and averaging convention before comparing F1 values directly.

## 4. Refit on all labelled data and write predictions

Refit the best feature-selected candidate from scratch on all rows of `train.csv`. This refits imputation, scaling, category encoding, and the selected Decision Tree-importance selector using training data only. Apply the fitted pipeline to `test.csv`, preserve test ID order, and write the same two columns as `sample_submission.csv`.

In [95]:
final_model = clone(candidate_models[best_model_name])
final_model.fit(X, y)
test_predictions = final_model.predict(test)

assert sample_submission.columns.tolist() == ["ID", TARGET]
assert sample_submission["ID"].equals(pd.Series(test.index, name="ID"))
assert len(test_predictions) == len(sample_submission)
assert set(np.unique(test_predictions)).issubset({0, 1})

submission = sample_submission.copy()
submission[TARGET] = test_predictions
submission_path = project_folder / "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Selected model: {best_model_name}")
print(f"Wrote {len(submission):,} predictions to {submission_path}")
display(submission.head())

Selected model: Logistic Regression + tree feature selection
Wrote 986 predictions to /home/yoru/Documents/novaims/Y1/S1/ML/ML_Project/submission.csv


,ID,repeat_purchase_90d
0,10000006,1
1,10000008,1
2,10000010,1
3,10000020,0
4,10000025,1
